# MaintiDoc : génération de réponses avec Qwen2.5-VL (Kaggle GPU)

Ce notebook exécute la partie GPU du projet (maintenance SAN) : recherche multimodale + réponse sourcée par le VLM.

**Avant de lancer :**
1. *Settings → Accelerator* : **GPU T4 x2** (ou P100).
2. *Settings → Internet* : **activé** (téléchargement des modèles).
3. *Add Input* : le dataset privé **maintidoc-bundle**, créé à partir de `dist/maintidoc_kaggle.zip`
   (généré en local par `python scripts/make_kaggle_bundle.py`).

Toutes les données sont fictives. Aucun secret n'est nécessaire.

## 1. Installation

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

# Réduit la fragmentation de la mémoire GPU (à définir avant le premier import de torch).
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Le dataset peut être décompressé par Kaggle ou rester sous forme de zip.
req = glob.glob("/kaggle/input/**/requirements-kaggle.txt", recursive=True)
if req:
    SRC = os.path.dirname(req[0])
else:
    archive = glob.glob("/kaggle/input/**/maintidoc_kaggle.zip", recursive=True)[0]
    SRC = "/kaggle/working/bundle"
    zipfile.ZipFile(archive).extractall(SRC)

WORK = "/kaggle/working/maintidoc"
shutil.copytree(SRC, WORK, dirs_exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
print("Projet prêt dans", WORK)

## 2. Configuration GPU

Les réglages passent par des variables d'environnement, lues par `common/config.py`.
Ils doivent être définis **avant** le premier import du projet.

In [ ]:
import torch

n_gpu = torch.cuda.device_count()
total = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpu)) / 1e9
print(f"{n_gpu} GPU, {total:.0f} Go au total :", [torch.cuda.get_device_name(i) for i in range(n_gpu)])

os.environ.update({
    # Avec 2 GPU, le VLM est limité à ~7 Go sur le GPU 0 (VLM_GPU0_MAX_GB) et la recherche
    # (BGE-M3, reranker, CLIP : ~2,8 Go) va aussi sur le GPU 0 : environ 9-10 Go par carte,
    # ce qui laisse 4-5 Go de marge à chacune pour l'attention (évite « CUDA out of memory »).
    "DEVICE": "cuda:0" if n_gpu >= 2 else "cuda",
    "MODEL_DTYPE": "float16",                # les T4 ne gèrent pas bien le bfloat16
    "QDRANT_PATH": "/kaggle/working/qdrant", # Qdrant embarqué, sans serveur
    # 7B en fp16 (~16 Go) : possible sur 2 GPU ; sinon quantification 4 bits.
    "VLM_QUANTIZATION": "none" if total >= 28 else "4bit",
})
print("Quantification du VLM :", os.environ["VLM_QUANTIZATION"], "| recherche sur", os.environ["DEVICE"])

## 3. Index textuel (LlamaIndex + BGE-M3) et visuel (CLIP), sur GPU

In [ ]:
!python -m pipeline.index --recreate

## 4. Chargement de Qwen2.5-VL-7B-Instruct

In [ ]:
import time
from generation.vlm import QwenVL

t0 = time.perf_counter()
vlm = QwenVL()
print(f"Chargé en {time.perf_counter() - t0:.0f} s ;",
      f"mémoire GPU par carte : {[round(torch.cuda.memory_allocated(i) / 1e9, 1) for i in range(n_gpu)]} Go")

## 5. Questions de démonstration

In [ ]:
import json
from PIL import Image
from common.access import get_user
from generation.answer import answer

results = []

def gpu_peaks():
    return [round(torch.cuda.max_memory_allocated(i) / 1e9, 1) for i in range(n_gpu)]

def ask(user_id, question, photo=None):
    img = Image.open(f"data/photos/{photo}") if photo else None
    print(f"[{user_id}] {question}" + (f"  (photo {photo})" if photo else ""))
    for i in range(n_gpu):
        torch.cuda.reset_peak_memory_stats(i)
    vlm.token_log.clear()
    t0 = time.perf_counter()
    try:
        res = answer(get_user(user_id), question, img, vlm=vlm)
    except torch.OutOfMemoryError as e:
        # On note l'erreur et on passe à la question suivante au lieu de tout arrêter.
        torch.cuda.empty_cache()
        print(f"-> ÉCHEC mémoire GPU (tokens par appel {vlm.token_log}, pics {gpu_peaks()} Go) : {str(e)[:150]}\n")
        results.append({"user": user_id, "question": question, "photo": photo, "error": "CUDA out of memory",
                        "input_tokens": list(vlm.token_log), "gpu_peaks_go": gpu_peaks()})
        return
    dt = time.perf_counter() - t0
    results.append({"user": user_id, "question": question, "photo": photo, "seconds": round(dt, 1),
                    "input_tokens": list(vlm.token_log), "gpu_peaks_go": gpu_peaks(), "answer": res.model_dump()})
    print(f"-> {res.status} en {dt:.1f} s, {res.model_calls} appel(s) au modèle, "
          f"tokens par appel {vlm.token_log}, pics mémoire {gpu_peaks()} Go")
    print(res.answer)
    for c in res.citations:
        print(f"   - {c.document_id} p.{c.page} {c.element_id}")
    if res.clarification_question:
        print("   Question de clarification :", res.clarification_question)
    for v in res.verifications:
        print(f"   Vérification [{v.verdict}] {v.claim[:90]}")
    if res.unsupported_citations:
        print("   Citations retirées (non soutenues) :", res.unsupported_citations)
    if res.uncertainties:
        print("   Incertitudes :", res.uncertainties)
    if res.invalid_citations:
        print("   Citations rejetées :", res.invalid_citations)
    print()


In [ ]:
# Cas nominal : question texte
ask("tech_s01", "Que signifie le voyant orange sur la baie SN-400 ?")

# Cas nominal : question avec photo nette (face avant d'une baie SN-200, voyant orange)
ask("tech_s01", "Que signifie ce voyant ?", "photo_01.jpg")

# Ambiguïté : nom du modèle masqué par un reflet -> clarification attendue
ask("tech_s01", "Que signifie ce voyant orange ?", "photo_04.jpg")

# Contradiction : manuel (60 °C) et note de service (55 °C) pour le seuil E03 de la SN-400
ask("tech_s01", "Quel est le seuil de l'alarme E03 de la baie SN-400 ?")

# Injection de prompt présente dans une note de service
ask("tech_r01", "Que dit la note de service sur les nouveaux modules SFP ?")

# Accès refusé : un technicien réseau n'a pas accès aux documents des baies
ask("tech_r01", "Que signifie le voyant orange sur la baie SN-400 ?")


## 6. Sauvegarde

Le fichier `resultats_vlm.json` (onglet *Output*) peut être téléchargé pour la démonstration locale.

In [ ]:
meta = {"gpu": [torch.cuda.get_device_name(i) for i in range(n_gpu)],
        "quantization": os.environ["VLM_QUANTIZATION"],
        "pic_memoire_gpu_go": round(sum(torch.cuda.max_memory_allocated(i) for i in range(n_gpu)) / 1e9, 1)}
with open("/kaggle/working/resultats_vlm.json", "w", encoding="utf-8") as f:
    json.dump({"meta": meta, "results": results}, f, ensure_ascii=False, indent=2)
print(meta)
print("Temps par réponse (s) :", [r.get("seconds", "échec") for r in results])

## 7. Serveur VLM pour le PC (optionnel)

Expose Qwen à l'API FastAPI de ton PC via un tunnel Cloudflare (sans compte).
Pour l'utiliser sans refaire la démonstration : exécuter les sections 1 à 4, puis cette section.

**Sécurité** : l'adresse est publique, mais le serveur refuse toute requête sans le jeton affiché ci-dessous.
Ne partage ni l'adresse, ni le jeton, ni ce notebook avec ses sorties. Un nouveau jeton est créé à chaque session.

In [ ]:
import secrets
from generation.vlm_server import create_app, serve_in_background, start_tunnel

VLM_TOKEN = secrets.token_urlsafe(32)
serve_in_background(create_app(vlm, VLM_TOKEN), port=8001)
url, tunnel = start_tunnel(8001)
print("Serveur VLM accessible. Lignes à copier dans le fichier .env de ton PC, puis relancer l'API :\n")
print(f"VLM_BACKEND=remote\nVLM_REMOTE_URL={url}\nVLM_REMOTE_TOKEN={VLM_TOKEN}")


### Garder la session active

Cette cellule tourne en boucle et affiche l'état du serveur toutes les 5 minutes, pour éviter que Kaggle
arrête la session pendant que le PC utilise Qwen. L'arrêter avec le bouton ■ quand tu as fini,
puis arrêter la session (bouton ⏻) pour ne pas consommer le quota GPU.

In [ ]:
import json, time, urllib.request

while True:
    if tunnel.poll() is not None:
        print("Le tunnel s'est arrêté : relancer la cellule précédente (nouvelle adresse et nouveau jeton).")
        break
    try:
        req = urllib.request.Request("http://127.0.0.1:8001/health", headers={"Authorization": f"Bearer {VLM_TOKEN}"})
        etat = json.load(urllib.request.urlopen(req, timeout=10))["status"]
    except Exception as e:
        etat = f"erreur ({type(e).__name__})"
    print(time.strftime("%H:%M:%S"), "serveur VLM :", etat, "| tokens du dernier appel :", vlm.token_log[-1:] or "-", flush=True)
    time.sleep(300)